# Importance Sampling

## Considere:

### I = ∫ [0, 1] 1/( 1 + x)² dx 

(a) Estime via Monte Carlo simples, usando a uniforme.  
(b) Escolha uma distribuição p(x).  
(c) Reescreva o estimador e calcule a integral.  
(d) Compare os métodos.  
(e) Justifique a escolha de p(x).  

<a href="https://colab.research.google.com/github/jgocana/Trabalhos-TP-547/blob/master/Trabalho%202/importance_sampling.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [ ]:
!pip install numpy
!pip install math
!pip install matplotlib

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

# ============================================================
# QUESTÃO 2 — Monte Carlo simples vs Importance Sampling
# I = ∫_0^1 1/(1+x)^2 dx
# ============================================================

def integral_exata_q2():
    """
    Calcula valor exato:
    ∫_0^1 1/(1+x)^2 dx = [-1/(1+x)]_0^1 = 1 - 1/2 = 1/2
    """
    return 0.5

def mc_simples_uniforme_q2(N, rng):
    """
    (a) Monte Carlo simples com X ~ Uniform(0,1):
        I_hat = (1/N) Σ g(Xi), onde g(x)=1/(1+x)^2
    """
    X = rng.random(N)
    g = 1.0 / (1.0 + X)**2
    return g.mean()

def amostrar_p_otima_q2(N, rng):
    """
    (b) Escolha p(x) = 2/(1+x)^2 em [0,1] (normalizada).
    Para usar importance sampling, precisamos amostrar de p.

    CDF:
      F(x) = ∫_0^x 2/(1+t)^2 dt
           = 2(1 - 1/(1+x))
           = 2x/(1+x)

    Inversa:
      u = 2x/(1+x)  ->  u(1+x)=2x -> u = x(2-u) -> x = u/(2-u)
    """
    U = rng.random(N)
    X = U / (2.0 - U)
    return X

def importance_sampling_q2(N, rng):
    """
    (c) Estimador importance sampling:
        I = E_p[ g(X)/p(X) ]
        g(x)=1/(1+x)^2
        p(x)=2/(1+x)^2

        g(x)/p(x) = (1/(1+x)^2) / (2/(1+x)^2) = 1/2 (constante!)
    Portanto:
        I_hat = média de (1/2) = 1/2 exatamente (variância zero).
    """
    X = amostrar_p_otima_q2(N, rng)

    g = 1.0 / (1.0 + X)**2
    p = 2.0 / (1.0 + X)**2

    w = g / p  # pesos
    return w.mean()

def questao2_comparacao(seed=20260406):
    """
    Executa:
    (a) MC simples uniforme
    (b) escolhe p(x)
    (c) importance sampling
    (d) compara erros (inclusive com repetições)
    (e) justifica p(x) (comentários no código e prints)
    """
    rng = np.random.default_rng(seed)
    I_exato = integral_exata_q2()

    Ns = [10**2, 10**3, 10**4, 10**5]

    print("\n=== QUESTÃO 2 ===")
    print(f"Valor exato: I = {I_exato:.6f}")

    # ---- Uma realização por N
    print("\n-- Uma realização por N --")
    for N in Ns:
        est_uni = mc_simples_uniforme_q2(N, rng)
        est_is  = importance_sampling_q2(N, rng)
        print(f"N={N:>7d} | MC simples={est_uni:.6f} | IS={est_is:.6f} | "
              f"erro_MC={abs(est_uni-I_exato):.6e} | erro_IS={abs(est_is-I_exato):.6e}")

    # ---- (d) Comparação mais robusta: várias repetições para ver variância
    R = 200  # número de repetições (ajuste se quiser)
    print(f"\n-- Comparação com R={R} repetições (mostra variância) --")

    for N in Ns:
        rng_rep = np.random.default_rng(seed + N)  # muda semente por N (reprodutível)
        est_uni_list = []
        est_is_list  = []

        for _ in range(R):
            est_uni_list.append(mc_simples_uniforme_q2(N, rng_rep))
            est_is_list.append(importance_sampling_q2(N, rng_rep))

        est_uni_list = np.array(est_uni_list)
        est_is_list  = np.array(est_is_list)

        # Erro médio absoluto e desvio padrão das estimativas
        mae_uni = np.mean(np.abs(est_uni_list - I_exato))
        mae_is  = np.mean(np.abs(est_is_list  - I_exato))
        std_uni = np.std(est_uni_list)
        std_is  = np.std(est_is_list)

        print(f"N={N:>7d} | MAE_MC={mae_uni:.3e} | MAE_IS={mae_is:.3e} | "
              f"STD_MC={std_uni:.3e} | STD_IS={std_is:.3e}")

# Executa
if __name__ == "__main__":
    questao2_comparacao()


=== QUESTÃO 2 ===
Valor exato: I = 0.500000

-- Uma realização por N --
N=    100 | MC simples=0.513087 | IS=0.500000 | erro_MC=1.308702e-02 | erro_IS=0.000000e+00
N=   1000 | MC simples=0.508142 | IS=0.500000 | erro_MC=8.141583e-03 | erro_IS=0.000000e+00
N=  10000 | MC simples=0.497273 | IS=0.500000 | erro_MC=2.726925e-03 | erro_IS=0.000000e+00
N= 100000 | MC simples=0.500211 | IS=0.500000 | erro_MC=2.114524e-04 | erro_IS=0.000000e+00

-- Comparação com R=200 repetições (mostra variância) --
N=    100 | MAE_MC=1.680e-02 | MAE_IS=0.000e+00 | STD_MC=2.104e-02 | STD_IS=0.000e+00
N=   1000 | MAE_MC=5.471e-03 | MAE_IS=0.000e+00 | STD_MC=6.690e-03 | STD_IS=0.000e+00
N=  10000 | MAE_MC=1.614e-03 | MAE_IS=0.000e+00 | STD_MC=2.014e-03 | STD_IS=0.000e+00
N= 100000 | MAE_MC=4.939e-04 | MAE_IS=0.000e+00 | STD_MC=6.225e-04 | STD_IS=0.000e+00
